# SIR-4 — retrieval baselines, all four domains

One notebook, four domains (**biology, cs, matsci, physics**), every row through the
**same scorer with the same flags**. Same arms, caveats and disclosures as the
single-domain baselines notebooks; read those in the per-domain header if this is the
first time through.

| family | arm | cost |
|---|---|---|
| lexical | BM25 | CPU, minutes per domain |
| dense | BGE-large, Qwen3-Embedding, SPECTER2-base, SciNCL | GPU, minutes each per domain |
| reasoning-trained dense | ReasonIR-8B | GPU, bf16, the slow dense arm |
| graph | G-Reasoner | **trained here per domain, on the v16sc graphs** — hours each |
| graph | GFM-RAG | **gated**: needs an OpenIE entity graph, which no SIR-4 domain has |

**Why G-Reasoner runs and GFM-RAG does not.** G-Reasoner's dataset class reads any typed
graph, so it trains on the v16sc frame graphs that already exist for all four domains.
GFM-RAG v1's forward path ranks `entity` nodes and maps them to documents
(`GraphIndexDatasetV1`, `target_type: entity`); the v16sc graphs have no entity nodes, and
SIR-4 has no OpenIE construction. Building one means LLM extraction over every document of
all four corpora. Until that exists, the GFM-RAG row is reported on TOMATO only, and
section 5d says so per domain rather than crashing.

**Everything is idempotent.** Dense arms skip on a matching manifest, training arms skip on
a matching signature, and all outputs persist to `outputs/baselines/sir4_<domain>/` — the
SAME namespace the single-domain notebooks use, so work done in either is visible to both.

**Slices are `all`, `same`, `cross` only**, for the same reason as the single-domain
notebooks: `similar`/`dissimilar` are defined by a reference dense run, which is circular
in a baseline table.

## 1. GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os, sys, json, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/cargo-gfmrag"
DOMS  = ["biology", "cs", "matsci", "physics"]        # stamped at build time from --dataset
DSETS = ["sir4_biology", "sir4_cs", "sir4_matsci", "sir4_physics"]
SPLIT = "test"
BUNDLES = {ds: f"{DRIVE}/{ds}_bundle.zip" for ds in DSETS}
# The SAME per-domain namespaces the single-domain baselines notebooks write, so a run
# finished in either notebook is a [skip] in the other, never a duplicate.
OUTRT = {ds: f"{DRIVE}/outputs/baselines/{ds}" for ds in DSETS}
CARGO_ROOT = "/content/cargo"
os.environ["CARGO_ROOT"] = CARGO_ROOT
S4 = f"{CARGO_ROOT}/sir4-retrieval"
for ds in DSETS:
    os.makedirs(OUTRT[ds], exist_ok=True)
_missing = [ds for ds in DSETS if not os.path.exists(BUNDLES[ds])]
for ds in DSETS:
    print(f"  {ds:14} bundle {'ok' if ds not in _missing else 'MISSING'}   -> {OUTRT[ds]}")
assert not _missing, f"missing bundles on Drive: {_missing}"

## 2. Unpack every bundle + install
One root, four bundles. Each bundle mirrors the repo, the data directories are dataset-name-scoped so they cannot collide, and the code files are identical copies, so last-unpacked wins harmlessly. **Full bundles required** for section 5: a `--slim` bundle carries no graphs.

In [ ]:
import zipfile, shutil, collections
if os.path.isdir(CARGO_ROOT):
    shutil.rmtree(CARGO_ROOT)
os.makedirs(CARGO_ROOT, exist_ok=True)
for ds in DSETS:
    zipfile.ZipFile(BUNDLES[ds]).extractall(CARGO_ROOT)
    print("unpacked", os.path.basename(BUNDLES[ds]))

# A code_overlay on Drive wins over every bundle's copies, so a script edited after the
# bundles were built does not need four re-uploads to take effect.
OV = f"{DRIVE}/code_overlay"
if os.path.isdir(OV):
    shutil.copytree(OV, CARGO_ROOT, dirs_exist_ok=True); print("applied code_overlay")

CORPUS = {ds: f"{CARGO_ROOT}/kg-construction/data/{ds}_{SPLIT}/raw" for ds in DSETS}
for ds in DSETS:
    for f in ("documents.json", f"{SPLIT}.json"):
        assert os.path.exists(f"{CORPUS[ds]}/{f}"), f"missing {CORPUS[ds]}/{f}"
    _c = json.load(open(f"{CORPUS[ds]}/documents.json"))
    _q = json.load(open(f"{CORPUS[ds]}/{SPLIT}.json"))
    _g = [len(x.get("supporting_documents") or []) for x in _q]
    print(f"  {ds:14} {len(_c):>7,} docs  {len(_q):>6,} queries  "
          f"golds/query {sum(_g)/len(_g):.2f}  "
          f"strata {dict(collections.Counter(x.get('stratum') for x in _q))}")
BL = f"{S4}/eval/baselines_sir4.py"
assert os.path.exists(BL), f"missing {BL} -- rebuild a bundle or use code_overlay"

# Same pinned install and the same reasoning as the single-domain notebooks: sections 1-4
# need sentence-transformers, and the pin must match section 5a's or whichever cell ran
# last decides the environment.
!pip -q install rank_bm25 sentence-transformers "transformers>=4.52.4,<5"
import transformers
print("ready | transformers", transformers.__version__)
assert transformers.__version__.startswith("4."), (
    f"transformers {transformers.__version__} is outside the supported 4.x range; "
    "restart the runtime so the pinned wheel is the one imported")

## 3. The arms
*(harvested verbatim from the single-domain builder — the two notebooks cannot define different baselines)*

In [ ]:
# Qwen3's STOCK retrieval instruction from its official model card. The newline is
# part of the format and must reach the tokenizer as a real newline, not the two
# characters backslash+n. The run cell below therefore passes argv as a list.
QWEN_INSTRUCT = ("Instruct: Given a web search query, retrieve relevant passages "
                  "that answer the query\nQuery:")
BGE_INSTRUCT = "Represent this sentence for searching relevant passages: "

# ReasonIR's documented default is an empty instruction. Keep the variable explicit so
# a future task-specific ReasonIR experiment cannot silently change the stock baseline.
REASONIR_INSTRUCT = ""

# (label, tag, model, pooling, instruct, extra flags)
ARMS = [
    ("BM25",            "bm25",     "bm25",                       "auto", "",           ""),
    ("BGE-large",       "bge",      "BAAI/bge-large-en-v1.5",     "st",   BGE_INSTRUCT, ""),
    ("Qwen3-Embedding", "qwen3",    "Qwen/Qwen3-Embedding-0.6B",  "st",   QWEN_INSTRUCT, ""),
    # CLS, NOT MEAN. Both are CLS-pooled; ST's mean-pooling fallback would evaluate a
    # different model than either paper and understate them.
    # SPECTER2-base, NOT SPECTER2. The published retrieval model is this base encoder PLUS
    # a task adapter (proximity for candidate papers, ad-hoc-query for short queries).
    # Loading the base alone is a different, weaker model, so the row is labelled for what
    # it is. To make it the real thing: pip install adapters, load allenai/specter2 onto
    # the base with load_as="proximity", which baselines_sir4.py does not yet support.
    ("SPECTER2-base",   "specter2", "allenai/specter2_base",      "cls",  "",           ""),
    # NO [SEP]. SciNCL and SPECTER2 are both trained on "title [SEP] abstract"; this corpus
    # is already flattened to "Title. Abstract" in documents.json and the title boundary is
    # not recoverable from it without the pre-flattening source. Both rows therefore see a
    # slightly different input format from the one their papers used. Same text, same
    # tokeniser, one missing separator token, and it applies equally to both rows, so it
    # cannot flip their comparison -- but it is a reason not to read either as that paper's
    # published number, and it belongs in the table caption.
    ("SciNCL",          "scincl",   "malteos/scincl",             "cls",  "",           ""),
    # 8B: half precision or it does not fit, and a custom architecture so it needs remote
    # code. BFLOAT16, not float16: the model card's own usage is torch_dtype="auto", which
    # resolves to the bf16 the checkpoint was trained in. Forcing fp16 re-quantises to a
    # format with a much smaller exponent range, which is the standard way an 8B scores
    # below its published numbers. bf16 needs Ampere or newer (A100 yes, T4 no).
    ("ReasonIR-8B",     "reasonir", "reasonir/ReasonIR-8B",       "st",   REASONIR_INSTRUCT,
     "--trust-remote-code --dtype bfloat16 --batch 8"),
]
for lab, tag, m, p, ins, extra in ARMS:
    print(f"  {lab:18} {m}")

## 4. Run every arm on every domain
Domain-major: all six arms for biology, then cs, and so on. Skips follow the same manifest rule as the single-domain notebooks, checked against the same files on Drive, so nothing already computed is recomputed.

In [ ]:
import hashlib, shlex
TOPK = 300
ACCEPT_UNVERIFIED = False

def sh(cmd, cwd=None):
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end="")
    return p.wait()

_SCORER = hashlib.md5(open(f"{S4}/eval/baselines_sir4.py", "rb").read()).hexdigest()[:8]

def arm_sig(ds, model, pool, ins, extra):
    # Identical fields to the single-domain notebooks' arm_sig, so the manifests they
    # wrote validate here and vice versa.
    return hashlib.md5(json.dumps(
        {"model": model, "pooling": pool, "instruct": ins, "extra": extra,
         "topk": TOPK, "scorer": _SCORER, "dataset": ds, "split": SPLIT},
        sort_keys=True).encode()).hexdigest()[:12]

PRED = {ds: {} for ds in DSETS}
for ds in DSETS:
    print(f"\n===================== {ds} =====================")
    os.environ["CARGO_DATASET"] = ds
    for lab, tag, model, pool, ins, extra in ARMS:
        dest = f"{S4}/data/predictions_{tag}_{ds}_{SPLIT}.json"
        man  = dest + ".manifest.json"
        sig  = arm_sig(ds, model, pool, ins, extra)
        PRED[ds][lab] = dest
        for a, b in ((f"{OUTRT[ds]}/{os.path.basename(dest)}", dest),
                     (f"{OUTRT[ds]}/{os.path.basename(man)}",  man)):
            if not os.path.exists(b) and os.path.exists(a):
                os.makedirs(os.path.dirname(b), exist_ok=True); shutil.copy(a, b)
        _have = None
        if os.path.exists(man):
            try: _have = json.load(open(man)).get("sig")
            except Exception: _have = None
        if os.path.exists(dest):
            if _have == sig:
                print(f"[skip] {ds}/{lab}: manifest matches"); continue
            if _have is None and ACCEPT_UNVERIFIED:
                print(f"[skip] {ds}/{lab}: NO MANIFEST, accepted (ACCEPT_UNVERIFIED)"); continue
            why = "no manifest" if _have is None else f"manifest {_have} != {sig}"
            print(f"[stale] {ds}/{lab}: {why} -- re-running")
        cmd = [sys.executable, "-u", "eval/baselines_sir4.py",
               "--dataset", ds, "--split", SPLIT, "--model", model,
               "--pooling", pool, "--tag", tag, "--topk", str(TOPK)]
        if ins:   cmd += ["--instruct", ins]
        if extra: cmd += shlex.split(extra)
        cmd += ["--out", dest]
        rc = sh(cmd, S4)
        if rc != 0:
            print(f"!! {ds}/{lab} FAILED rc={rc} -- row reported as missing")
        else:
            json.dump({"sig": sig, "model": model, "pooling": pool, "instruct": ins,
                       "extra": extra, "topk": TOPK, "scorer_md5": _SCORER},
                      open(man, "w"), indent=1)
            shutil.copy(dest, f"{OUTRT[ds]}/{os.path.basename(dest)}")
            shutil.copy(man,  f"{OUTRT[ds]}/{os.path.basename(man)}")

## 5. Graph baseline — G-Reasoner per domain

**Everything below is optional and slow.** Sections 1-4 are the dense table; stop there if
that is all you need today.

G-Reasoner (`GraphReasoner`, stock `sft_training` config) trains from random init on each
domain's **v16sc graphs**, which the full bundles already carry. First run per domain also
builds the Qwen3 node index (~20-45 min); training is hours per domain at batch 2, and the
loop is domain-serial with signature skips, so a disconnect costs only the run in flight.

GFM-RAG is section 5d and is **gated**: it prints what it needs instead of crashing.

### 5a. Engine
*(reused verbatim from the fusion notebook)*

In [ ]:
import os, sys, torch
!rm -rf /content/gfm-rag
!cd /content && unzip -q {DRIVE}/gfm-rag-adapted.zip
!pip install -q --no-deps -e /content/gfm-rag
# TORCH IS DELIBERATELY NOT IN THIS LIST. Colab ships a torch/torchvision pair built
# against each other, and asking pip for `torch` can move torch off the version its
# torchvision was compiled for. THAT mismatch is what produced
#   ImportError: cannot import name 'VideoReader'
# from datasets' torch formatter. gfmrag installs --no-deps, so nothing here needs a
# torch newer than the host image's.
!pip install -q torch-geometric sentence-transformers transformers hydra-core omegaconf \
              easydict ninja faiss-cpu pymetis wandb tqdm numpy pandas python-dotenv \
              langchain-community 2>&1 | tail -3

# REPAIR, NEVER REMOVE. This used to be `pip uninstall -y torchvision`, on the reasoning
# that gfmrag does not use it. That reasoning has expired: current transformers resolves
# PreTrainedModel through a lazy module that imports torchvision, so deleting it turns
# every `from transformers import ...` into
#   ModuleNotFoundError: Could not import module 'PreTrainedModel'
# and takes sentence-transformers, and therefore every encoder in this notebook, down with
# it. Verify the pair instead, and only intervene if it is actually broken.
import torch
try:
    import torchvision
    from transformers import PreTrainedModel          # the import that has to work
    print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | transformers ok")
except Exception as _e:
    # STOP, DO NOT SELF-HEAL. The obvious repair, `pip install torchvision`, resolves to
    # the LATEST torchvision and drags torch up with it: measured on 2026-08-17 it took a
    # stock runtime from torch 2.11.0+cu128 to 2.13.0+cu130, a 2 GB download that rebuilt
    # the whole CUDA stack, broke Colab's cudf/cuml/raft pins, and left the running kernel
    # holding the OLD torch. Silently re-pinning CUDA under a training run is far worse
    # than refusing, because the damage only surfaces as `torch.cuda.is_available()` going
    # False, or as numerics nobody can reproduce.
    #
    # A matched pair is what the stock image already ships. Getting back to it is one
    # menu action, and no pip incantation is more reliable than that.
    raise RuntimeError(
        f"torchvision/transformers are broken in this runtime "
        f"({type(_e).__name__}: {_e}).\n"
        f"torch here is {torch.__version__}.\n"
        f"FIX: Runtime > Disconnect and DELETE runtime (not 'Restart session' -- a restart "
        f"keeps whatever pip did), then run this notebook from the top. The stock image "
        f"ships a matched torch/torchvision pair and this cell installs neither, so the "
        f"check above will pass with no downloads.\n"
        f"Do NOT `pip install torchvision` to get past this: it upgrades torch and CUDA "
        f"underneath you."
    ) from _e

# THE OTHER HALF OF THE TORCHVISION STORY. datasets' torch formatter runs
# `from torchvision.io import VideoReader` whenever torchvision is importable, and
# current torchvision has REMOVED VideoReader. That import sits on the training
# dataloader's hot path, so with torchvision present every training run dies on its
# first batch. Uninstalling torchvision was the old workaround and now breaks
# transformers instead (see above), so the remaining move is to patch datasets ON
# DISK: guard the import, skip the isinstance when it is unavailable. The training
# subprocess re-imports datasets from disk, so the patch reaches it with no restart.
import re as _re
import datasets.formatting.torch_formatter as _dtf
_p = _dtf.__file__
_s = open(_p).read()
if "VideoReader = None" in _s:
    print("datasets torch formatter already patched")
else:
    _s2 = _re.sub(r'^( *)from torchvision\.io import VideoReader$',
                  lambda m: (f"{m.group(1)}try:\n{m.group(1)}    from torchvision.io import VideoReader\n"
                             f"{m.group(1)}except Exception:\n{m.group(1)}    VideoReader = None"),
                  _s, flags=_re.M)
    _s2 = _s2.replace("isinstance(value, VideoReader)",
                      "(VideoReader is not None and isinstance(value, VideoReader))")
    assert _s2 != _s, "VideoReader import not found -- datasets layout changed, patch by hand"
    open(_p, "w").write(_s2)
    print("patched datasets torch formatter:", _p)
# Replay the exact failing path in a fresh interpreter: torchvision imported (that is
# what arms the buggy branch), then a torch-formatted Dataset read.
import subprocess as _sp
_r = _sp.run([sys.executable, "-c",
              "import torchvision, datasets\n"
              "d = datasets.Dataset.from_dict({'x': [1, 2, 3]}).with_format('torch')\n"
              "print('datasets formatter ok:', d[:2]['x'])"],
             capture_output=True, text=True)
print(_r.stdout.strip())
assert _r.returncode == 0, _r.stderr[-2000:]

def soft_import(path, line, fallback):
    t = open(path).read()
    if f"try:\n    {line}" not in t:
        open(path, "w").write(t.replace(line, f"try:\n    {line}\nexcept Exception:\n    {fallback}"))
soft_import("/content/gfm-rag/gfmrag/text_emb_models/__init__.py",
            "from .qwen3_model import Qwen3TextEmbModel", "Qwen3TextEmbModel = None")
# pylate/ColBERT entity-linker is unused by SFT training; make its import non-fatal so the
# training subprocess (fresh Python re-imports gfmrag) does not crash on `import pylate`.
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/entity_linking_model/__init__.py",
            "from .colbert_el_model import ColbertELModel", "ColbertELModel = None")
# 4c. LLM-OpenIE model imports langchain_community (ChatOllama/ChatLlamaCpp); the installed version
#     dropped ChatOllama. SFT training never builds an index, so make this import non-fatal.
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/openie_model/__init__.py",
            "from .llm_openie_model import LLMOPENIEModel", "LLMOPENIEModel = None")
# 4d. same for the LLM-NER model (separate __init__, separate import line).
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/ner_model/__init__.py",
            "from .llm_ner_model import LLMNERModel", "LLMNERModel = None")
# the adapted zip strips config/wandb/ (and sometimes config/text_emb_model/) -> create before writing
for _cd in ["/content/gfm-rag/gfmrag/workflow/config/wandb",
            "/content/gfm-rag/gfmrag/workflow/config/text_emb_model"]:
    os.makedirs(_cd, exist_ok=True)
open("/content/gfm-rag/gfmrag/workflow/config/wandb/default.yaml", "w").write(
    'enabled: false\nlog_model: false\nproject: "gfm-rag"\nentity: null\nname: null\ngroup: null\ntags: []\nnotes: ""\n')
open("/content/gfm-rag/gfmrag/workflow/config/text_emb_model/qwen3_st.yaml", "w").write(
    '_target_: gfmrag.text_emb_models.BaseTextEmbModel\n'
    'text_emb_model_name: /content/qwen3\nnormalize: True\nbatch_size: 32\n'   # LOCAL path, not hub name (see cell 3)
    'query_instruct: "Instruct: Given a scientific research problem or open need, retrieve papers whose method, mechanism, or technique could be borrowed as inspiration, including transfers from other domains.\\nQuery: "\n'
    'passage_instruct: null\nmodel_kwargs: null\n')
sys.path.insert(0, "/content/gfm-rag")
# stub the unused pylate/ColBERT dep so the IN-KERNEL gfmrag import does not crash
import types as _t
for _m in ["pylate", "pylate.indexes", "pylate.models", "pylate.retrieve"]:
    sys.modules.setdefault(_m, _t.ModuleType(_m))
class _D:
    def __init__(self, *a, **k): pass
sys.modules["pylate.indexes"].PLAID = _D; sys.modules["pylate.models"].ColBERT = _D; sys.modules["pylate.retrieve"].ColBERT = _D
from gfmrag.models.gfm_reasoner import GraphReasoner
assert torch.cuda.is_available(), "Use an A100/high-RAM GPU"
print("G-Reasoner OK |", torch.cuda.get_device_name(0))

# ---- PATCH: per-epoch STRATIFIED metrics ----
# Training runs in a subprocess, so we edit the source: wrap trainer.evaluate() to add
# per-slice document_hits@k/mrr keys. _log_metrics then prints them EACH EPOCH in the same
# format as the aggregate lines. Toggle with STRAT_EVAL=0; BGE split from STRAT_BGE.
STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
_src = open(STF).read()
if "_evaluate_stratified" not in _src:
    _inject = "\n".join([
        "    # --- injected: per-epoch stratified eval (ZERO extra forward pass) ---",
        "    # A forward hook records each eval query's gold-document rank during evaluate()'s",
        "    # existing pass; we then add per-slice keys to the metrics dict (logged as usual).",
        "    import os as _o, json as _j",
        "    from collections import defaultdict as _dd",
        "    _rec = []",
        "    _recording = {'on': False}",
        "    def _hook(_module, _inp, _out):",
        "        if not _recording['on'] or len(_inp) < 2:",
        "            return",
        "        try:",
        "            _g, _b = _inp[0], _inp[1]",
        "            _did = _g.nodes_by_type['document']",
        "            _dp = _out[:, _did]",
        "            _tgt = _b['target_nodes_mask'][:, _did].bool()",
        "            _rk = _dp.argsort(dim=-1, descending=True).argsort(dim=-1)",
        "            _ids = _b['id']",
        "            for _qi in range(_dp.shape[0]):",
        "                _pos = _tgt[_qi].nonzero(as_tuple=True)[0]",
        "                if len(_pos):",
        "                    _r = int(_rk[_qi, _pos].min().item()) + 1",
        "                    _q = _ids[_qi]",
        "                    _q = _q.item() if hasattr(_q, 'item') else _q",
        "                    _rec.append((_q, _r))",
        "        except Exception:",
        "            pass",
        "    trainer.model.register_forward_hook(_hook)",
        "    _orig_evaluate = trainer.evaluate",
        "    def _evaluate_stratified():",
        "        _rec.clear(); _recording['on'] = True",
        "        m = _orig_evaluate()",
        "        _recording['on'] = False",
        "        if _o.environ.get('STRAT_EVAL','1') != '1':",
        "            return m",
        "        try:",
        "            _name = _o.environ.get('STRAT_NAME','eval')",
        "            _tj = _o.environ.get('STRAT_TEST','')",
        "            _bp = _o.environ.get('STRAT_BGE','')",
        "            _meta = {q['id']: q for q in _j.load(open(_tj))} if _tj and _o.path.exists(_tj) else {}",
        "            _BGE = {r['id']: r for r in _j.load(open(_bp))} if _bp and _o.path.exists(_bp) else {}",
        "            def _brank(qid, g):",
        "                for i,(d,_s) in enumerate(_BGE.get(qid,{}).get('predictions',{}).get('document',[]),1):",
        "                    if d==g: return i",
        "                return 10**9",
        "            _sl = _dd(lambda: _dd(list)); _seen = set()",
        "            for _q,_r in _rec:",
        "                if _q in _seen: continue",
        "                _seen.add(_q)",
        "                _mq = _meta.get(_q, {})",
        "                _gg = _mq.get('supporting_documents') or []",
        "                _gd = _gg[0] if isinstance(_gg,list) and _gg else _gg",
        "                _st = 'same' if _mq.get('stratum')=='same' else 'cross'",
        "                _sim = 'dissim' if _brank(_q,_gd)>100 else 'sim'",
        "                for _nm in [_st,_sim] + (['cross+dissim'] if (_st=='cross' and _sim=='dissim') else []):",
        "                    _d = _sl[_nm]",
        "                    for _k in (1,5,10): _d['hits@'+str(_k)].append(float(_r<=_k))",
        "                    _d['mrr'].append(1.0/_r if _r<=100 else 0.0)",
        "            for _nm,_d in _sl.items():",
        "                _n = len(_d['mrr']) or 1",
        "                for _c in ('hits@1','hits@5','hits@10','mrr'):",
        "                    m[_name+'/document_'+_c+'/'+_nm] = sum(_d[_c])/_n",
        "        except Exception as _e:",
        "            print('[stratified] skipped:', _e)",
        "        return m",
        "    trainer.evaluate = _evaluate_stratified",
        "    trainer.train()",
    ])
    _src = _src.replace("    trainer.train()", _inject, 1)
    open(STF, "w").write(_src)
    print("patched sft_training.py -> per-epoch stratified eval")
else:
    print("stratified patch already applied")
# ---- PATCH: tqdm shows per-component RUNNING-AVERAGE losses (bce / pcr / mse / total) ----
# Patches base_trainer.py ON DISK so the training SUBPROCESS shows every loss, not just the total.
_bt = "/content/gfm-rag/gfmrag/trainers/base_trainer.py"
_bs = open(_bt).read()
_OLD = 'progress_bar.set_postfix(loss=step_metrics.get("loss", 0.0))'
_NEW = ('_names = {"bce_loss": "bce", "pcr_loss": "pcr", "mse_loss": "mse", "loss": "tot"}\n'
        '                progress_bar.set_postfix({_names.get(k, k): f"{np.mean(v):.3f}" for k, v in epoch_metrics.items()})')
if "_names.get(k, k)" in _bs:
    print("base_trainer.py already patched (per-component postfix present)")
elif _OLD in _bs:
    open(_bt, "w").write(_bs.replace(_OLD, _NEW))
    print("patched base_trainer.py -> tqdm shows running-average bce/pcr/mse/tot")
else:
    print("WARN: postfix line not found in base_trainer.py (file may have changed); inspect ~line 424")


### 5a-i. Re-pin the two packages the engine installs unbounded

In [ ]:
# The engine is 4.x-era transformers and 0.18.x-era wandb. An unpinned install here has
# already produced transformers 5.x with wandb 0.28 in this notebook.
!pip -q install "transformers>=4.52.4,<5" "wandb>=0.18.5,<0.19"
# Check what is INSTALLED, not what this kernel has imported. The training subprocess is a
# fresh python and sees the installed versions; the kernel may already hold a stale wandb
# module from an earlier cell (an import of transformers/sentence-transformers can pull
# wandb in), which made this assert fire on 0.28.1 in the combined MIR notebook although
# pip had installed 0.18.x correctly.
import importlib.metadata as _im
_tv, _wv = _im.version("transformers"), _im.version("wandb")
print("installed: transformers", _tv, "| wandb", _wv)
assert _tv.startswith("4."), _tv
assert _wv.startswith("0.18."), _wv
# Both asserts, not a print. A silent mismatch here surfaces much later as an unrelated
# traceback inside the training subprocess, which is the worst place to debug it.
import sys as _sys
for _m, _v in (("transformers", _tv), ("wandb", _wv)):
    _loaded = getattr(_sys.modules.get(_m), "__version__", None)
    if _loaded and _loaded != _v:
        print(f"note: this kernel imported {_m} {_loaded} earlier; the subprocess will use {_v}")
print("if either assert fired, restart the runtime and re-run this cell before 5b")

In [ ]:
import os, shutil
QDIR = "/content/qwen3"
DRIVE_QWEN = f"{DRIVE}/qwen3-embedding-0.6b"
BASE = "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main"
TOK = os.environ.get("HF_TOKEN", "")
AUTH = f'-H "Authorization: Bearer {TOK}"' if TOK else ""
NEED = ["model.safetensors","config.json","config_sentence_transformers.json","modules.json",
        "tokenizer.json","tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]

def ready(d):
    return (all(os.path.exists(f"{d}/{f}") for f in NEED)
            and os.path.getsize(f"{d}/model.safetensors") > 1_000_000_000
            and os.path.getsize(f"{d}/tokenizer.json") > 11_000_000)

if not ready(QDIR) and ready(DRIVE_QWEN):
    print("restoring Qwen3 from Drive cache ..."); shutil.copytree(DRIVE_QWEN, QDIR, dirs_exist_ok=True)

if not ready(QDIR):
    os.makedirs(f"{QDIR}/1_Pooling", exist_ok=True)
    # 1) big weight via aria2c (run ONCE — a repeat can delete the finished file)
    if not (os.path.exists(f"{QDIR}/model.safetensors") and os.path.getsize(f"{QDIR}/model.safetensors") > 1_000_000_000):
        os.system("apt-get -qq install -y aria2")
        os.system(f'aria2c -x16 -s16 -k1M --max-tries=5 --retry-wait=2 --file-allocation=none '
                  f'{AUTH} -d {QDIR} -o model.safetensors "{BASE}/model.safetensors"')
    # 2) small files bypass Xet — plain curl
    for f in ["config.json","config_sentence_transformers.json","modules.json",
              "tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]:
        os.system(f'curl -sSL -f {AUTH} "{BASE}/{f}" -o "{QDIR}/{f}"')
    # 3) tokenizer.json (11 MB, also Xet) — retry until a request lands on the good CDN
    for i in range(20):
        os.system(f"rm -f {QDIR}/tokenizer.json")
        os.system(f'curl -sSL -f {AUTH} "{BASE}/tokenizer.json" -o {QDIR}/tokenizer.json')
        if os.path.exists(f"{QDIR}/tokenizer.json") and os.path.getsize(f"{QDIR}/tokenizer.json") > 11_000_000:
            print(f"tokenizer.json ok on try {i+1}"); break
    assert ready(QDIR), "Qwen3 incomplete — re-run this cell (aria2c may need another pass)"
    os.makedirs(os.path.dirname(DRIVE_QWEN), exist_ok=True)
    shutil.copytree(QDIR, DRIVE_QWEN, dirs_exist_ok=True); print("cached Qwen3 to Drive")

# load by LOCAL PATH — never by hub name again
from sentence_transformers import SentenceTransformer
_m = SentenceTransformer(QDIR)
print("Qwen3 loaded offline:", _m.encode(["test"], normalize_embeddings=True).shape)  # (1, 1024)
del _m

### 5a-ii. Fix the vendored PyG version check

In [ ]:
# The ULTRA layers vendored in the engine parse the PyG version as:
#     pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]
# Colab resolves torch-geometric to versions like 2.6.1.post1, and int("post1") raises
# ValueError inside the first message-passing call. Taking the first three dotted
# components and keeping only the numeric ones handles "2.6.1.post1" and
# "2.7.0+pt24cu121" alike, and unlike a version pin it will not drift at the next release.
import os, torch_geometric
OLD = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
NEW = ('pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3] '
       'if i.isdigit()]')
hits = []
for root, _, files in os.walk("/content/gfm-rag/gfmrag"):
    for f in files:
        if not f.endswith(".py"):
            continue
        fp = os.path.join(root, f)
        t = open(fp).read()
        if OLD in t:
            open(fp, "w").write(t.replace(OLD, NEW))
            hits.append(fp)
parsed = [int(i) for i in torch_geometric.__version__.split(".")[:3] if i.isdigit()]
if hits:
    for h in hits:
        print("  patched", h)
else:
    # IDEMPOTENT. A bare `assert hits` failed on every re-run of this cell, because the
    # first run already replaced the only occurrences. Absent is fine as long as the
    # CORRECTED form is what is there instead; absent with neither form present means the
    # engine changed and the patch would be silently doing nothing.
    _already = [os.path.join(r, f)
                for r, _, fs in os.walk("/content/gfm-rag/gfmrag") for f in fs
                if f.endswith(".py") and NEW in open(os.path.join(r, f)).read()]
    assert _already, ("neither the original nor the corrected version check is present -- "
                      "the engine changed, do not run on an unpatched copy")
    for h in _already:
        print("  already patched", h)
print(f"torch_geometric {torch_geometric.__version__} now parses to {parsed}")

### 5a-iii. Restore `torchvision.io.VideoReader` for `datasets`

In [ ]:
# THE TRAINING SUBPROCESS IS A FRESH PYTHON. `python -m gfmrag.workflow.sft_training` does
# not inherit this kernel's patched modules, so the shim has to live in a file that the
# subprocess imports. sft_training.py is that file, and is already patched twice below.
import torch, torchvision, torchvision.io, datasets
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"datasets {datasets.__version__} | VideoReader "
      f"{hasattr(torchvision.io, 'VideoReader')}")

_SHIM = """# PATCH (notebook): datasets' torch formatter imports torchvision.io.VideoReader
# whenever torchvision is in sys.modules. Recent torchvision removed the legacy video API, so
# that import raises inside every batch fetch. The name is only needed for an isinstance
# check against tensor data, so a placeholder is sufficient and downloads nothing.
import torchvision.io as _tvio
if not hasattr(_tvio, "VideoReader"):
    class _NoVideoReader:
        def __init__(self, *a, **k):
            raise RuntimeError("torchvision video API removed; this shim exists only so "
                               "the datasets torch formatter can import the name")
    _tvio.VideoReader = _NoVideoReader
"""

_STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
_t = open(_STF).read()
if "_NoVideoReader" in _t:
    print("[skip] sft_training.py already carries the shim")
else:
    # PREPENDED, not injected at an anchor. The other two patches of this file search for a
    # line inside main(); this one must run before `datasets` is imported anywhere, so it goes
    # above every import. Prepending also leaves their anchors untouched.
    open(_STF, "w").write(_SHIM + _t)
    print("shimmed", _STF)

# This kernel too: the in-kernel gfmrag import and any in-notebook dataset use hit the same
# formatter.
exec(_SHIM)
print("VideoReader present now:", hasattr(torchvision.io, "VideoReader"))

# SECOND ENGINE FIX, same cell so the ALL-mode harvest carries both. The zip's
# base_trainer calls _load_checkpoint from _setup_model BEFORE the AMP scaler is built,
# so any resume_from_checkpoint dies on self.scaler.load_state_dict with a bare
# AttributeError. The optimizer load directly above it already guards with hasattr;
# this line was missed. Skipping is harmless: bf16 runs a DISABLED GradScaler, so the
# saved scaler state is empty anyway.
_bt = "/content/gfm-rag/gfmrag/trainers/base_trainer.py"
_t = open(_bt).read()
_OLDS = 'self.scaler.load_state_dict(state["scaler"])'
if 'hasattr(self, "scaler")' in _t:
    print("[skip] base_trainer scaler guard already present")
else:
    assert _OLDS in _t, "scaler load line not found -- engine changed, inspect by hand"
    _t = _t.replace(_OLDS,
        'self.scaler.load_state_dict(state["scaler"]) if hasattr(self, "scaler") else '
        'logger.warning("resume: scaler not built yet, skipped scaler state")', 1)
    open(_bt, "w").write(_t)
    print("patched base_trainer.py scaler guard (resume path)")

# THIRD ENGINE FIX, the other half of the resume path. _setup_model loads the checkpoint
# BEFORE the bf16 cast, so Optimizer.load_state_dict casts Adam's exp_avg to the params'
# then-current fp32; after the cast the first step mixes fp32 state with bf16 grads and
# dies in _foreach_lerp_. Move the load to after precision setup, matching the dtype
# layout a fresh run creates lazily.
_t = open(_bt).read()
_EARLY = ("        if self.args.resume_from_checkpoint:\n"
          "            self._load_checkpoint(self.args.resume_from_checkpoint)\n")
_SCALER = ("        self.scaler = torch.amp.GradScaler(\n"
           "            self.device.type, enabled=self.enable_grad_scaler\n"
           "        )\n")
if _t.index(_EARLY) > _t.index(_SCALER):
    print("[skip] resume already loads after precision setup")
else:
    assert _t.count(_EARLY) == 1 and _t.count(_SCALER) == 1, "resume/scaler anchors moved"
    _t = _t.replace(_EARLY, "", 1)
    _t = _t.replace(_SCALER, _SCALER +
        "        # PATCH (notebook): resume moved after the precision cast; loading\n"
        "        # earlier poisons Adam state dtype and the first step dies.\n" + _EARLY, 1)
    open(_bt, "w").write(_t)
    print("patched base_trainer.py resume-after-precision order")

### 5b. Ship repo files + check every domain's graphs

In [ ]:
import json, os, csv, collections, shutil
FILES = json.loads(r'''{"/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_gfmrag.yaml": "# GFM-RAG v1 as a BASELINE: the published architecture, trained from random init on our\n# graph and split. No pretrained GFM-RAG weights.\n#\n# This is the repo's own reference config (config/gfm_rag/sft_training.yaml) with only the\n# adaptations our data forces. It is NOT derived from sft_training.yaml (G-Reasoner); an\n# earlier version of this file was, and claimed in a comment to differ from it in two\n# places, which was false in four.\n#\n# WHY THIS RUNS ON tomato_train / tomato_test AND NOT ON v16sc.\n#   GNNRetriever.forward ends in map_entities_to_docs, which dereferences\n#   graph.target_to_other_types (model.py:318). Only GraphIndexDatasetV1 sets that\n#   attribute, and V1 needs target_type: entity. Our v16sc graph has no `entity` nodes at\n#   all -- its node types are limitation/method/function/mechanism/task/finding/domain +\n#   document -- so target_type: entity finds zero targets there. tomato_{train,test} is the\n#   OpenIE construction, typed entity + document, which is exactly what this model expects.\n#\n# ADAPTATIONS, all forced by the data, each one deliberate:\n#   text_emb_model  mpnet -> qwen3, and feat_dim 768 -> 1024, because that is what the\n#                   graph was indexed with. A baseline reading different features from the\n#                   arms it is compared against is not a controlled row.\n#   train/valid     our splits.\n# Everything else -- dataset class, target_type, ranker, losses, lr, entity_model width --\n# is the reference recipe.\n#\n# entity_model WIDTH STAYS AT THE REFERENCE 512. feat_dim 1024 does NOT propagate into the\n# GNN: the model projects the 1024-d features down to the GNN width itself, at\n#   rel_mlp      = nn.Linear(feat_dim, entity_model.dims[0])    model.py:59\n#   question_mlp = nn.Linear(feat_dim, entity_model.dims[0])    model.py:60\n# so feat_dim and the hidden width are independent. An earlier version of this file widened\n# the six layers to 1024 \"to match feat_dim\", which was not required and roughly quadrupled\n# the layer parameter count -- a baseline with more capacity than the published architecture\n# is not that architecture.\n#\n# CACHE SAFETY. processed_dir is {root}/{name}/processed/stage2/{fingerprint}, and the\n# fingerprint is md5(class_name + text_emb_cfgs + {use_node_feat, use_relation_feat,\n# use_edge_feat, inverse_relation_feat}). The class name differs from G-Reasoner's, so the\n# two arms' graph.pt files land in different directories and cannot overwrite each other.\nhydra:\n  run:\n    dir: outputs/qa_finetune/${now:%Y-%m-%d}/${now:%H-%M-%S}\n  searchpath:\n    - pkg://gfmrag.workflow.config\n\ndefaults:\n  - _self_\n  # GFM-RAG v1's own default. SimpleRanker would report a weaker model than the paper's\n  # under the paper's name; plain idf_ranker is the un-truncated variant, and the\n  # reference uses the top-k one.\n  - doc_ranker: idf_topk_ranker\n  - text_emb_model: qwen3\n  - wandb: default\n\nseed: 1024\ntimeout: 60\nsave_pretrained: no\nload_model_from_pretrained: null\n\ndatasets:\n  # V1, not GraphIndexDataset. Required, not preferred: see the forward-path note above.\n  _target_: gfmrag.graph_index_datasets.GraphIndexDatasetV1\n  cfgs:\n    root: ./data\n    force_reload: False\n    text_emb_model_cfgs: ${text_emb_model}\n    target_type: entity          # entity reasoning, then entity->document ranking\n    use_node_feat: False         # GFM-RAG v1 does not use node features\n    use_edge_feat: False         # nor edge features\n    use_relation_feat: True\n    inverse_relation_feat: text  # v1 prefixes \"inverse\" to relation names\n  train_names:\n    - tomato_train\n  valid_names:\n    - tomato_test\n  init_datasets: True\n  feat_dim: 1024\n  max_datasets_in_memory: 10\n  data_loading_workers: 4\n\nmodel:\n  _target_: gfmrag.models.gfm_rag_v1.GNNRetriever\n  ranker: ${doc_ranker}\n  init_nodes_weight: True\n  # MUST be set whenever init_nodes_weight is True: model.py:268 asserts on it. The\n  # previous version of this file left it null with weighting on, which raised on the\n  # first forward pass.\n  init_nodes_type: document\n  dtype: bfloat16\n  entity_model:\n    # Reference width. Do NOT raise these to feat_dim: rel_mlp/question_mlp already\n    # project 1024 -> dims[0]. See the note in the header.\n    _target_: gfmrag.models.ultra.models.QueryNBFNet\n    input_dim: 512\n    hidden_dims: [512, 512, 512, 512, 512, 512]\n    message_func: distmult\n    aggregate_func: sum\n    short_cut: yes\n    layer_norm: yes\n\n# Reference supervision: ENTITY nodes. Document scores come out of the ranker, and the\n# document metrics below are computed on them, so this row stays directly comparable with\n# G-Reasoner's document metrics even though the loss sits at a different level.\n#\n# Entity-level supervision is what makes this GFM-RAG rather than \"GNNRetriever trained\n# like G-Reasoner\". To run the matched-recipe variant instead, change both\n# target_node_type below to `document` and add G-Reasoner's MSE distillation term -- but\n# then the row must not be labelled GFM-RAG.\nlosses:\n  - name: bce_loss\n    loss:\n      _target_: gfmrag.losses.BCELoss\n      adversarial_temperature: 0.2\n    weight: 0.3\n    target_node_type: entity\n  - name: pcr_loss\n    loss:\n      _target_: gfmrag.losses.ListCELoss\n    weight: 0.7\n    target_node_type: entity\n\noptimizer:\n  _target_: torch.optim.AdamW\n  lr: 5.0e-4\n\ntrainer:\n  _target_: gfmrag.trainers.SFTTrainer\n  args:\n    _target_: gfmrag.trainers.TrainingArguments\n    train_batch_size: 8\n    num_epoch: 20\n    logging_steps: 100\n    max_steps_per_epoch: null\n    resume_from_checkpoint: null\n    do_train: true\n    do_eval: true\n    save_best_only: yes\n    # Same selection metric as the G-Reasoner arm, so neither arm is chosen on a\n    # different criterion from the other. valid_names is the test set, so this keeps the\n    # best test epoch; both arms are selected identically, so the comparison between them\n    # is unaffected.\n    metric_for_best_model: document_mrr\n    dtype: ${model.dtype}\n    split_graph_inference: false\n    split_graph_training: false\n    split_graph_partition: contiguous\n  metrics:\n    - mrr\n    - ndcg@5\n    - recall@3\n    - recall@5\n  target_types:\n    - entity\n    - document\n", "/content/gfm-rag/gfmrag/utils/qa_utils.py": "# mypy: ignore-errors\n\nimport torch\nfrom torch import distributed as dist\n\nfrom gfmrag.models.ultra import variadic\n\n\nclass DocumentRetriever:\n    \"\"\"\n    Return documents based on document ranking\n    \"\"\"\n\n    def __init__(self, docs: dict, id2doc: dict) -> None:\n        self.docs = docs\n        self.id2doc = id2doc\n\n    def __call__(self, doc_ranking: torch.Tensor, top_k: int = 1) -> list:\n        top_k_docs = doc_ranking.topk(top_k).indices\n        norm_doc_scors = mini_max_scale(doc_ranking)\n        return [\n            {\n                \"title\": self.id2doc[doc.item()],\n                \"content\": self.docs[self.id2doc[doc.item()]],\n                \"score\": doc_ranking[doc].item(),\n                \"norm_score\": norm_doc_scors[doc].item(),\n            }\n            for doc in top_k_docs\n        ]\n\n\ndef mini_max_scale(tensor):\n    return (tensor - tensor.min()) / (tensor.max() - tensor.min())\n\n\ndef entities_to_mask(entities, num_nodes):\n    mask = torch.zeros(num_nodes)\n    mask[entities] = 1\n    return mask\n\n\ndef evaluate(pred, target, metrics):\n    ranking, num_pred = pred\n    answer_ranking, num_hard = target\n    answer_ranking = answer_ranking + 1\n    metric = {}\n    for _metric in metrics:\n        if _metric == \"mrr\":\n            answer_score = 1 / ranking.float()\n            query_score = variadic.variadic_mean(answer_score, num_hard)\n        elif _metric.startswith(\"recall@\"):\n            threshold = int(_metric[7:])\n            answer_score = (answer_ranking <= threshold).float()\n            query_score = (\n                variadic.variadic_sum(answer_score, num_hard) / num_hard.float()\n            )\n        elif _metric.startswith(\"hits@\"):\n            threshold = int(_metric[5:])\n            answer_score = (ranking <= threshold).float()\n            query_score = variadic.variadic_mean(answer_score, num_hard)\n        elif _metric.startswith(\"ndcg@\"):\n            # Binary-relevance nDCG, identical to score_sir4.py's definition, so the\n            # per-epoch curve and the reported table are the same quantity. Uses\n            # answer_ranking (position in the ranked list) rather than the filtered\n            # `ranking`, because nDCG is about where a gold actually landed.\n            threshold = int(_metric[5:])\n            gain = torch.where(\n                answer_ranking <= threshold,\n                1.0 / torch.log2(answer_ranking.float() + 1.0),\n                torch.zeros_like(answer_ranking, dtype=torch.float),\n            )\n            dcg = variadic.variadic_sum(gain, num_hard)\n            # Ideal DCG: min(num_golds, k) golds sitting at positions 1..n. Built as a\n            # cumulative table and indexed, so queries with different gold counts each\n            # get their own ceiling and a 2-gold query is not penalised for having\n            # fewer golds than k.\n            disc = 1.0 / torch.log2(\n                torch.arange(1, threshold + 1, device=gain.device).float() + 1.0\n            )\n            idcg_table = torch.cat(\n                [torch.zeros(1, device=gain.device), disc.cumsum(0)]\n            )\n            idcg = idcg_table[num_hard.clamp(max=threshold)]\n            query_score = dcg / idcg.clamp(min=1e-9)\n        elif _metric == \"mape\":\n            query_score = (num_pred - num_hard).abs() / (num_hard).float()\n        else:\n            raise ValueError(f\"Unknown metric `{_metric}`\")\n\n        score = query_score.mean()\n        name = _metric\n        metric[name] = score.item()\n\n    return metric\n\n\ndef gather_results(pred, target, rank, world_size, device):\n    # for multi-gpu setups: join results together\n    # for single-gpu setups: doesn't do anything special\n    ranking, num_pred = pred\n    answer_ranking, num_target = target\n\n    all_size_r = torch.zeros(world_size, dtype=torch.long, device=device)\n    all_size_ar = torch.zeros(world_size, dtype=torch.long, device=device)\n    all_size_p = torch.zeros(world_size, dtype=torch.long, device=device)\n    all_size_r[rank] = len(ranking)\n    all_size_ar[rank] = len(answer_ranking)\n    all_size_p[rank] = len(num_pred)\n    if world_size > 1:\n        dist.all_reduce(all_size_r, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_size_ar, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_size_p, op=dist.ReduceOp.SUM)\n\n    # obtaining all ranks\n    cum_size_r = all_size_r.cumsum(0)\n    cum_size_ar = all_size_ar.cumsum(0)\n    cum_size_p = all_size_p.cumsum(0)\n\n    all_ranking = torch.zeros(all_size_r.sum(), dtype=torch.long, device=device)\n    all_num_pred = torch.zeros(all_size_p.sum(), dtype=torch.long, device=device)\n    all_answer_ranking = torch.zeros(all_size_ar.sum(), dtype=torch.long, device=device)\n    all_num_target = torch.zeros(all_size_p.sum(), dtype=torch.long, device=device)\n\n    all_ranking[cum_size_r[rank] - all_size_r[rank] : cum_size_r[rank]] = ranking\n    all_num_pred[cum_size_p[rank] - all_size_p[rank] : cum_size_p[rank]] = num_pred\n    all_answer_ranking[cum_size_ar[rank] - all_size_ar[rank] : cum_size_ar[rank]] = (\n        answer_ranking\n    )\n    all_num_target[cum_size_p[rank] - all_size_p[rank] : cum_size_p[rank]] = num_target\n\n    if world_size > 1:\n        dist.all_reduce(all_ranking, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_num_pred, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_answer_ranking, op=dist.ReduceOp.SUM)\n        dist.all_reduce(all_num_target, op=dist.ReduceOp.SUM)\n\n    return (all_ranking.cpu(), all_num_pred.cpu()), (\n        all_answer_ranking.cpu(),\n        all_num_target.cpu(),\n    )\n\n\ndef batch_evaluate(pred, target, limit_nodes=None):\n    num_target = target.sum(dim=-1)\n\n    # answer2query = functional._size_to_index(num_answer)\n    answer2query = torch.repeat_interleave(num_target)\n\n    num_entity = pred.shape[-1]\n\n    # in inductive (e) fb_ datasets, the number of nodes in the graph structure might exceed\n    # the actual number of nodes in the graph, so we'll mask unused nodes\n    if limit_nodes is not None:\n        # print(f\"Keeping only {len(limit_nodes)} nodes out of {num_entity}\")\n        keep_mask = torch.zeros(num_entity, dtype=torch.bool, device=limit_nodes.device)\n        keep_mask[limit_nodes] = 1\n        # keep_mask = F.one_hot(limit_nodes, num_entity)\n        pred[:, ~keep_mask] = float(\"-inf\")\n\n    order = pred.argsort(dim=-1, descending=True)\n\n    range = torch.arange(num_entity, device=pred.device)\n    ranking = variadic.native_scatter(\n        range.expand_as(order), order, dim=-1, reduce=\"sum\"\n    )\n\n    target_ranking = ranking[target]\n    # unfiltered rankings of all answers\n    order_among_answer = variadic.variadic_sort(target_ranking, num_target)[1]\n    order_among_answer = (\n        order_among_answer + (num_target.cumsum(0) - num_target)[answer2query]\n    )\n\n    ranking_among_answer = variadic.native_scatter(\n        variadic.variadic_arange(num_target), order_among_answer, reduce=\"sum\"\n    )\n\n    # filtered rankings of all answers\n    ranking = target_ranking - ranking_among_answer + 1\n    ends = num_target.cumsum(0)\n    starts = ends - num_target\n    hard_mask = variadic.multi_slice_mask(starts, ends, ends[-1])\n    # filtered rankings of hard answers\n    ranking = ranking[hard_mask]\n\n    return ranking, target_ranking\n"}''')
for p, c in FILES.items():
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w').write(c)
    print('wrote', p)

csv.field_size_limit(10 ** 7)
DATA_ROOT = f'{CARGO_ROOT}/kg-construction/data'
ENTITY_OK = {}
for ds in DSETS:
    # v16sc graphs: required for the G-Reasoner arm. The loader reads
    # {graph}/raw/documents.json, a copy of the corpus INSIDE the graph directory.
    for split in ('train', 'test'):
        g  = f'{ds}_{split}_v16sc'
        s1 = f'{DATA_ROOT}/{g}/processed/stage1'
        assert os.path.exists(f'{s1}/nodes.csv'), (
            f'missing {s1}/nodes.csv -- the {ds} bundle is --slim, section 5 needs the '
            f'FULL bundle')
        src = f'{DATA_ROOT}/{ds}_{split}/raw/documents.json'
        dst = f'{DATA_ROOT}/{g}/raw/documents.json'
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
    # OpenIE entity graph: the GFM-RAG gate. Exists for TOMATO, for no SIR-4 domain.
    s1 = f'{DATA_ROOT}/{ds}_test/processed/stage1'
    if os.path.exists(f'{s1}/nodes.csv'):
        _t = collections.Counter(r['type'] for r in csv.DictReader(open(f'{s1}/nodes.csv')))
        ENTITY_OK[ds] = _t.get('entity', 0) > 0 and _t.get('document', 0) > 0
    else:
        ENTITY_OK[ds] = False
    _e = 'present' if ENTITY_OK[ds] else 'absent -> GFM-RAG gated'
    print(f'  {ds:14} v16sc graphs ok   entity graph: {_e}')

### 5c. Train G-Reasoner on every domain
Signature-gated exactly like the single-domain notebook: a finished run of this configuration is a `[skip]`, a directory produced by a different one is refused rather than adopted.

In [ ]:
EPOCHS, BATCH = 10, 2
import threading

def _sync_dir(src, dst):
    """Copy new/changed files src -> dst, tolerating a dead mount. Returns note."""
    try:
        for root, _, files in os.walk(src):
            rel = os.path.relpath(root, src)
            os.makedirs(os.path.join(dst, rel) if rel != "." else dst, exist_ok=True)
            for f in files:
                s = os.path.join(root, f)
                d = os.path.join(dst, rel, f) if rel != "." else os.path.join(dst, f)
                if (not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s)
                        or os.path.getmtime(s) > os.path.getmtime(d) + 1):
                    shutil.copy2(s, d)
        return "ok"
    except OSError as e:
        return f"skipped ({e})"

def run_baseline(ds, config, suffix, train, valid, epochs=EPOCHS, batch=BATCH):
    """Train LOCALLY, sync to Drive every 10 min. Drive is never on the training hot
    path: a FUSE mount flap killed a run at epoch 9 of 10 through the console-log tee,
    and the trainer's own checkpoint saves were one flap away from the same fate. Now a
    flap costs one sync pass and a runtime disconnect costs <=10 min of training.
    Skip/stale/resume decisions read the DRIVE copy, which is the durable one."""
    drive_dir = f"{OUTRT[ds]}/{ds}_{suffix}"
    run_dir   = f"/content/runs/{ds}_{suffix}"
    _ckpt_d, _pred_d = f"{drive_dir}/model_best.pth", f"{drive_dir}/predictions_{valid}.json"
    _cfgp = f"/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/{config}.yaml"
    _sig = hashlib.md5(json.dumps(
        {"cfg": hashlib.md5(open(_cfgp, "rb").read()).hexdigest(),
         "epochs": epochs, "batch": batch, "topk": TOPK,
         "train": train, "valid": [valid]}, sort_keys=True).encode()).hexdigest()[:12]
    _was = None
    if os.path.isfile(f"{drive_dir}/arm.json"):
        try: _was = json.load(open(f"{drive_dir}/arm.json")).get("sig")
        except Exception: _was = None
    if os.path.isfile(_ckpt_d) and os.path.isfile(_pred_d):
        if _was == _sig:
            print(f"[skip] {drive_dir}: finished run of this configuration")
            return drive_dir
        print(f"[stale] {drive_dir}: sig mismatch -- NOT overwriting and NOT reporting it")
        return None
    os.makedirs(run_dir, exist_ok=True); os.makedirs(drive_dir, exist_ok=True)
    cmd = [sys.executable, "-u", "-m", "gfmrag.workflow.sft_training",   # the kernel's python, where gfmrag is installed; a bare `python` resolved to /usr/bin/python3 once (8 Sep)
           "--config-path", "config/gfm_reasoner", "--config-name", config,
           "text_emb_model=qwen3_st", f"datasets.cfgs.root={DATA_ROOT}",
           "datasets.cfgs.force_reload=False",
           f"datasets.train_names=[{train}]", f"datasets.valid_names=[{valid}]",
           f"trainer.args.num_epoch={epochs}", f"trainer.args.train_batch_size={batch}",
           "+trainer.args.do_predict=true", f"+trainer.args.predict_top_k={TOPK}",
           f"hydra.run.dir={run_dir}"]
    # RESUME, not redo. The checkpoint carries epoch and optimizer state, and selection
    # is best-epoch anyway, so resuming from best loses at most the epochs since it.
    if os.path.isfile(_ckpt_d) and _was == _sig:
        print("[resume] restoring checkpoint from Drive and resuming from it")
        shutil.copy(_ckpt_d, f"{run_dir}/model_best.pth")
        cmd.append(f"trainer.args.resume_from_checkpoint={run_dir}/model_best.pth")
    elif os.path.isfile(_ckpt_d):
        print("[redo] Drive checkpoint is from a DIFFERENT configuration -- from scratch")
    env = dict(os.environ, WANDB_MODE="disabled", HYDRA_FULL_ERROR="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", STRAT_EVAL="0",
               # the engine must be importable from /content in a fresh interpreter; the
               # editable install did not guarantee that on the Sept 2026 image
               PYTHONPATH="/content/gfm-rag" + os.pathsep + os.environ.get("PYTHONPATH", ""))
    for k in ("OPERATOR_COMPONENTS", "OPERATOR_COMPONENTS_TEST", "SEMANTIC_COMPONENTS",
              "SEMANTIC_COMPONENTS_TEST", "SEMANTIC_CKPT", "SEMANTIC_POPNET",
              "FUSION_OBJECTIVE", "FUSION_ROUTER", "FUSION_GAMMAFIX", "HARDNEG_HUB",
              "HARDNEG_RAND", "HARDNEG_GRAPH", "PER_GOLD", "AUX_W", "SEM_POP_LAMBDA",
              "CCMP", "CCMP_W", "CCMP_LR", "MISS_W_AUX", "RESID_PRIOR", "CQIG_M"):
        env.pop(k, None)
    json.dump({"arm": suffix, "config": config, "epochs": epochs, "batch": batch,
               "baseline": True, "sig": _sig, "train": train, "valid": [valid],
               "predict_top_k": TOPK}, open(f"{run_dir}/arm.json", "w"), indent=1)
    print(f"[baseline] {ds}: {config} -> {run_dir}  (syncing to {drive_dir} every 10 min)")
    _stop = threading.Event()
    def _pump():
        while not _stop.wait(600):
            print(f"[sync] {_sync_dir(run_dir, drive_dir)}")
    _t = threading.Thread(target=_pump, daemon=True); _t.start()
    try:
        with open(f"{run_dir}/console.log", "w") as log:
            p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end=""); log.write(line)
            rc = p.wait()
    finally:
        # Runs on crash too, so the last checkpoint reaches Drive either way.
        _stop.set()
        print(f"[sync-final] {_sync_dir(run_dir, drive_dir)}")
    assert rc == 0, f"{ds}/{suffix} failed, exit code {rc} (-9 = killed, out of memory)"
    return drive_dir

for ds in DSETS:
    rd = run_baseline(ds, "sft_training", f"greasoner_e{EPOCHS}_b{BATCH}",
                      f"{ds}_train_v16sc", f"{ds}_test_v16sc")
    if rd:
        p = f"{rd}/predictions_{ds}_test_v16sc.json"
        if os.path.exists(p):
            PRED[ds]["G-Reasoner"] = p

### 5d. GFM-RAG — gated on an OpenIE entity graph

GFM-RAG v1 ranks `entity` nodes and maps them to documents, so it needs the OpenIE
construction (`entity` + `document` node types). **No SIR-4 domain has one**; building it
means LLM extraction (NER + OpenIE triples) over every document of the corpus, then the
GFM-RAG index build. Until then this section reports the gate per domain and the thesis
reports GFM-RAG on TOMATO, where the entity graph exists.

If an entity graph is ever built for a domain (`<ds>_{train,test}/processed/stage1` with
entity-typed nodes in the bundle), this cell picks it up on the next run with no edits.

In [ ]:
for ds in DSETS:
    if not ENTITY_OK.get(ds):
        print(f"[gate] {ds}: no OpenIE entity graph -- GFM-RAG row not trainable on this "
              f"domain (see the section header); reported on TOMATO only")
        continue
    rd = run_baseline(ds, "sft_training_gfmrag", f"gfmrag_e{EPOCHS}_b{BATCH}",
                      f"{ds}_train", f"{ds}_test")
    if rd:
        p = f"{rd}/predictions_{ds}_test.json"
        if os.path.exists(p):
            PRED[ds]["GFM-RAG"] = p

## 6. Score every arm on every domain, same scorer, same flags
CompleteSet@5 is the one SIR-4-specific column: 1 if a complete inspiration set from `sets.json` sits inside the top 5. It is requested ONLY when the domain's sets.json is present, because score_sir4 writes a hard 0.0 for the column otherwise, which reads as measured-and-zero.

In [ ]:
# `map` is in the list for MIR's table (R@3 / R@5 / nDCG@5 / mAP); harmless for SIR-4.
STD_COLS = ("mrr", "ndcg@5", "recall@3", "recall@5", "recall@10", "recall@25", "recall@100", "map")
GRID_KS  = (1, 3, 5, 10, 20, 25, 50, 100)
_BASE_COLS = ",".join(dict.fromkeys(list(STD_COLS)
                              + [f"recall@{k}" for k in GRID_KS]
                              + [f"hits@{k}" for k in GRID_KS]))
# Stamped at build time from the same table run_domain.py stages from.
SETS = {ds: f"{CARGO_ROOT}/{rel}" for ds, rel in {"sir4_biology": "quartet/data/benchmark/biology_test_low/sets.json", "sir4_cs": "quartet/data/benchmark/cs_test_final/sets.json", "sir4_matsci": "quartet/data/benchmark/matsci_test_low/sets.json", "sir4_physics": "quartet/data/benchmark/physics_test_low/sets.json"}.items()}
SCORES = {ds: {} for ds in DSETS}
for ds in DSETS:
    QS = f"{CORPUS[ds]}/{SPLIT}.json"
    _has_sets = os.path.exists(SETS.get(ds, ""))
    COLS = _BASE_COLS + (",completeset@5" if _has_sets else "")
    if not _has_sets:
        print(f"{ds}: sets.json MISSING at {SETS.get(ds)} -- CompleteSet@5 skipped, "
              f"not zero-filled")
    for lab, p in list(PRED[ds].items()):
        if not p or not os.path.exists(p):
            print(f"skipping {ds}/{lab}: no predictions"); continue
        jo = f"{OUTRT[ds]}/scores_{lab.replace(' ', '_').replace('/', '-')}.json"
        _extra = f" --sets {SETS[ds]}" if _has_sets else ""
        rc = sh(f"python3 -u eval/score_sir4.py --pred {p} --queries {QS} --cols {COLS}"
                f"{_extra} --name '{ds} {lab}' --json-out {jo}", S4)
        if rc == 0 and os.path.exists(jo):
            SCORES[ds][lab] = json.load(open(jo))
    print(f"{ds}: scored {sorted(SCORES[ds])}")

## 7. The tables

Per-domain standard metrics first (same columns and order as every fusion and ablation
notebook), then the cross-domain summary grid: one row per arm, one column pair per
domain, at the cutoffs in `CUTOFFS`. `drop` is `mean_k (1 - cross@k / same@k)`.

In [ ]:
CUTOFFS = (1, 10, 50)
PAIR    = ("same", "cross")
METRIC  = "recall"
assert all(k in GRID_KS for k in CUTOFFS), (
    f"CUTOFFS {CUTOFFS} not a subset of GRID_KS {GRID_KS}; add the cutoff and re-run 6")
ORDER = [l for l, *_ in ARMS] + ["G-Reasoner", "GFM-RAG"]

for ds in DSETS:
    print(f"\n================ {ds} ================")
    print("\nSTANDARD METRICS (%)")
    for slc in ("all", "same", "cross"):
        print(f"\n--- {slc} ---")
        _cols = list(STD_COLS) + ["completeset@5"]
        print(f"{'arm':22}{'n':>6}" + "".join(
            f"{c.replace('recall@', 'R@').replace('completeset@5', 'CS@5'):>9}" for c in _cols))
        for lab in ORDER:
            sc = (SCORES[ds].get(lab) or {}).get(slc)
            if not sc:
                continue
            print(f"{lab:22}{sc.get('n', 0):>6}"
                  + "".join(f"{100*sc[c]:>9.1f}" if sc.get(c) is not None else f"{'--':>9}"
                            for c in _cols))

L, R = PAIR
print(f"\n\n================ CROSS-DOMAIN SUMMARY ({METRIC}, %) ================")
for k in CUTOFFS:
    print(f"\n--- {METRIC}@{k}: ({L} / {R}) per domain ---")
    print(f"{'arm':22}" + "".join(f"{d:>16}" for d in DOMS) + f"{'mean drop':>12}")
    for lab in ORDER:
        row, drops, have = f"{lab:22}", [], False
        for ds in DSETS:
            sc = SCORES[ds].get(lab) or {}
            a = (sc.get(L) or {}).get(f"{METRIC}@{k}")
            c = (sc.get(R) or {}).get(f"{METRIC}@{k}")
            if a is None or c is None:
                row += f"{'--':>16}"
            else:
                row += f"{100*a:>8.1f}/{100*c:<7.1f}"; have = True
                if a: drops.append(1.0 - c / a)
        row += (f"{-100*sum(drops)/len(drops):>10.0f}%" if drops else f"{'--':>12}")
        if have:
            print(row)

for ds in DSETS:
    json.dump(SCORES[ds], open(f"{OUTRT[ds]}/baselines_all.json", "w"), indent=1)
json.dump({ds: SCORES[ds] for ds in DSETS},
          open(f"{DRIVE}/outputs/baselines/sir4_all_domains.json", "w"), indent=1)
print(f"\nwrote per-domain baselines_all.json + {DRIVE}/outputs/baselines/sir4_all_domains.json")